# Multimodal and Document AI, Text-to-SQL

Notebook for **Chapter 12** of the *genai* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/genai/ch12.html

Tip: change a number, predict the output, then run the cell again.

## 12.1 How Models See: Images, Patches and Vision-Language Models

In [ ]:
import math, numpy as np
def patches(h,w,p): return math.ceil(h/p)*math.ceil(w/p)
print("224x224, p16:", patches(224,224,16))
print("1024x768, p16:", patches(1024,768,16))
print("1024x768, p32:", patches(1024,768,32))
for p in (16,32,64): print("A4 300dpi p%d:"%p, patches(3508,2480,p))
# toy CLIP-style zero-shot: embeddings are made up, NOT from a real model
rng=np.random.default_rng(0)
labels=["a photo of a cat","a photo of a dog","a screenshot of a chart"]
T=rng.normal(size=(3,8)); T/=np.linalg.norm(T,axis=1,keepdims=True)
img=T[2]+0.3*rng.normal(size=8); img/=np.linalg.norm(img)
sims=T@img
for l,s in zip(labels,sims): print("%-26s cos=%.3f"%(l,s))
print("predicted:",labels[int(np.argmax(sims))])

224x224, p16: 196
1024x768, p16: 3072
1024x768, p32: 768
A4 300dpi p16: 34100
A4 300dpi p32: 8580
A4 300dpi p64: 2145
a photo of a cat           cos=-0.162
a photo of a dog           cos=0.364
a screenshot of a chart    cos=0.846
predicted: a screenshot of a chart


## 12.2 Document AI: OCR, OCR-Free Models and Structured Extraction

In [ ]:
import json, re
from datetime import datetime
def fake_vlm(attempt):
    if attempt==1:
        return '{"invoice_no":"INV-204","date":"05/01/2025","items":[{"desc":"Cable","qty":2,"price":150.0},{"desc":"Router","qty":1,"price":2500.0}],"total":2900.0}'
    return '{"invoice_no":"INV-204","date":"2025-01-05","items":[{"desc":"Cable","qty":2,"price":150.0},{"desc":"Router","qty":1,"price":2500.0}],"total":2800.0}'
def validate(raw):
    errs=[]
    try: d=json.loads(raw)
    except json.JSONDecodeError as e: return None,["not JSON: "+str(e)]
    if not re.fullmatch(r"INV-\d+",str(d.get("invoice_no",""))): errs.append("bad invoice_no")
    try: datetime.strptime(d.get("date",""),"%Y-%m-%d")
    except ValueError: errs.append("date must be YYYY-MM-DD, got %r"%d.get("date"))
    calc=sum(i["qty"]*i["price"] for i in d.get("items",[]))
    if abs(calc-d.get("total",-1))>0.01: errs.append("total %s != sum of items %s"%(d.get("total"),calc))
    return d,errs
for attempt in (1,2):
    d,errs=validate(fake_vlm(attempt)); print("attempt",attempt,"errors:",errs)
    if not errs: print("accepted:",d["invoice_no"],d["date"],d["total"]); break

attempt 1 errors: ["date must be YYYY-MM-DD, got '05/01/2025'", 'total 2900.0 != sum of items 2800.0']
attempt 2 errors: []
accepted: INV-204 2025-01-05 2800.0


## 12.3 Speech: ASR, TTS, Diarization and Word Error Rate

In [ ]:
def wer(ref,hyp):
    r,h=ref.split(),hyp.split()
    d=[[0]*(len(h)+1) for _ in range(len(r)+1)]
    for i in range(len(r)+1): d[i][0]=i
    for j in range(len(h)+1): d[0][j]=j
    for i in range(1,len(r)+1):
        for j in range(1,len(h)+1):
            c=0 if r[i-1]==h[j-1] else 1
            d[i][j]=min(d[i-1][j]+1,d[i][j-1]+1,d[i-1][j-1]+c)
    return d[-1][-1],len(r),d[-1][-1]/len(r)
print(wer("the server restarted at nine pm","the server restart at nine"))
print(wer("refund within thirty days","refund within thirty days"))
print(wer("pay five thousand rupees","pay fifty thousand rupees now"))

(2, 6, 0.3333333333333333)
(0, 4, 0.0)
(2, 4, 0.5)


## 12.4 Text-to-SQL: Building an Analytics Assistant

In [ ]:
import sqlite3, re
con=sqlite3.connect(":memory:")
con.executescript("""
create table orders(id integer primary key, customer text, amount real, created date);
insert into orders values (1,'asha',120.0,'2025-01-05'),(2,'ravi',80.5,'2025-01-20'),(3,'asha',45.0,'2025-02-02'),(4,'meena',200.0,'2025-02-11');
""")
def fake_text2sql(q):
    if "total" in q and "customer" in q: return "SELECT customer, SUM(amount) AS total FROM orders GROUP BY customer ORDER BY total DESC"
    if "delete" in q: return "DELETE FROM orders"
    if "salary" in q: return "SELECT salary FROM employees"
    return "SELECT COUNT(*) FROM orders"
FORBIDDEN=re.compile(r"\b(insert|update|delete|drop|alter|create|attach|pragma|replace)\b",re.I)
def guard(sql):
    s=sql.strip().rstrip(";")
    if ";" in s: raise ValueError("multiple statements")
    if not re.match(r"(?i)select\b",s): raise ValueError("only SELECT allowed")
    if FORBIDDEN.search(s): raise ValueError("forbidden keyword")
    return s+" LIMIT 100" if "limit" not in s.lower() else s
def ask(q):
    sql=fake_text2sql(q); print("Q:",q,"\n SQL:",sql)
    try:
        g=guard(sql); rows=con.execute(g).fetchall(); print(" rows:",rows)
    except Exception as e: print(" REJECTED/ERROR:",type(e).__name__,e)
ask("total amount per customer"); ask("please delete everything"); ask("average salary"); ask("how many orders")

Q: total amount per customer 
 SQL: SELECT customer, SUM(amount) AS total FROM orders GROUP BY customer ORDER BY total DESC
 rows: [('meena', 200.0), ('asha', 165.0), ('ravi', 80.5)]
Q: please delete everything 
 SQL: DELETE FROM orders
 REJECTED/ERROR: ValueError only SELECT allowed
Q: average salary 
 SQL: SELECT salary FROM employees
 REJECTED/ERROR: OperationalError no such table: employees
Q: how many orders 
 SQL: SELECT COUNT(*) FROM orders
 rows: [(4,)]


In [ ]:
import sqlite3, tempfile, os
# read-only connection (needs a file database; ':memory:' cannot be reopened read-only)
path=os.path.join(tempfile.mkdtemp(),"demo.db")
rw=sqlite3.connect(path); rw.execute("create table t(x)"); rw.commit(); rw.close()
ro=sqlite3.connect(f"file:{path}?mode=ro",uri=True)
try: ro.execute("insert into t values (1)")
except sqlite3.OperationalError as e: print("read-only db:",e)

read-only db: attempt to write a readonly database


In [ ]:
import sqlite3
con=sqlite3.connect(":memory:")
con.executescript("""
create table orders(id integer primary key, customer text, amount real, created date);
insert into orders values (1,'asha',120.0,'2025-01-05'),(2,'ravi',80.5,'2025-01-20'),(3,'asha',45.0,'2025-02-02'),(4,'meena',200.0,'2025-02-11');
create table staff(name text, salary real);
insert into staff values ('x',1000);
""")
ALLOWED={"orders"}
def auth(action,a1,a2,dbname,src):
    if action==sqlite3.SQLITE_SELECT: return sqlite3.SQLITE_OK
    if action==sqlite3.SQLITE_READ: return sqlite3.SQLITE_OK if a1 in ALLOWED else sqlite3.SQLITE_DENY
    if action==sqlite3.SQLITE_FUNCTION: return sqlite3.SQLITE_OK
    return sqlite3.SQLITE_DENY
con.set_authorizer(auth)
for q in ["SELECT SUM(amount) FROM orders","SELECT salary FROM staff","DELETE FROM orders","SELECT * FROM orders WHERE 1=1; DROP TABLE orders"]:
    try: print(q,"->",con.execute(q).fetchall())
    except Exception as e: print(q,"-> BLOCKED:",type(e).__name__,e)

SELECT SUM(amount) FROM orders -> [(445.5,)]
SELECT salary FROM staff -> BLOCKED: DatabaseError access to staff.salary is prohibited
DELETE FROM orders -> BLOCKED: DatabaseError not authorized
SELECT * FROM orders WHERE 1=1; DROP TABLE orders -> BLOCKED: ProgrammingError You can only execute one statement at a time.


## 12.5 Copilots and Analytics Assistants: Putting the Pieces Together

## 12.6 Edit Distance and WER by Hand

In [ ]:
def table(ref, hyp):
    r, h = ref.split(), hyp.split()
    d = [[0] * (len(h) + 1) for _ in range(len(r) + 1)]
    for i in range(len(r) + 1): d[i][0] = i
    for j in range(len(h) + 1): d[0][j] = j
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            c = 0 if r[i - 1] == h[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + c)
    return r, h, d

def counts(ref, hyp):
    r, h, d = table(ref, hyp)
    i, j, S, D, I, ops = len(r), len(h), 0, 0, 0, []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i][j] == d[i - 1][j - 1] + (r[i - 1] != h[j - 1]):
            if r[i - 1] != h[j - 1]:
                S += 1; ops.append("sub %s->%s" % (r[i - 1], h[j - 1]))
            i -= 1; j -= 1
        elif i > 0 and d[i][j] == d[i - 1][j] + 1:
            D += 1; ops.append("del " + r[i - 1]); i -= 1
        else:
            I += 1; ops.append("ins " + h[j - 1]); j -= 1
    return S, D, I, len(r), ops[::-1]

r, h, d = table("set the alarm for seven", "set an alarm seven")
print("      " + " ".join("%6s" % w for w in ["-"] + h))
for i, row in enumerate(d):
    print("%-6s" % (["-"] + r)[i] + " ".join("%6d" % x for x in row))
S, D, I, N, ops = counts("set the alarm for seven", "set an alarm seven")
print("S,D,I,N =", S, D, I, N, "| edits:", ops, "| WER =", (S + D + I) / N)

           -    set     an  alarm  seven
-          0      1      2      3      4
set        1      0      1      2      3
the        2      1      1      2      3
alarm      3      2      2      1      2
for        4      3      3      2      2
seven      5      4      4      3      2
S,D,I,N = 1 1 0 5 | edits: ['sub the->an', 'del for'] | WER = 0.4


In [ ]:
import re
def wer(ref, hyp):
    r, h = ref.split(), hyp.split()
    d = list(range(len(h) + 1))
    for i in range(1, len(r) + 1):
        prev, d[0] = d[0], i
        for j in range(1, len(h) + 1):
            cur = min(d[j] + 1, d[j - 1] + 1, prev + (r[i - 1] != h[j - 1]))
            prev, d[j] = d[j], cur
    return d[-1] / len(r)

def norm(s):
    s = s.lower().replace(",", "")
    return re.sub(r"[^a-z0-9 ]", "", s)

ref = "Dr. Rao paid Rs. 5,000 today"
hyp = "doctor rao paid rs 5000 today"
print("raw WER       :", round(wer(ref, hyp), 3))
print("normalised WER:", round(wer(norm(ref), norm(hyp)), 3))

raw WER       : 0.667
normalised WER: 0.167


## 12.7 Attacking Your Own SQL Guard

In [ ]:
import re
FORBIDDEN = re.compile(r"\b(insert|update|delete|drop|alter|create|attach|pragma|replace)\b", re.I)
def guard(sql):
    s = sql.strip().rstrip(";")
    if ";" in s: raise ValueError("multiple statements")
    if not re.match(r"(?i)select\b", s): raise ValueError("only SELECT allowed")
    if FORBIDDEN.search(s): raise ValueError("forbidden keyword")
    return s + " LIMIT 100" if "limit" not in s.lower() else s

cases = [
    ("plain select", "SELECT * FROM orders"),
    ("stacked", "SELECT 1; DROP TABLE orders"),
    ("lower-case", "delete from orders"),
    ("word in a string", "SELECT * FROM orders WHERE customer='drop'"),
    ("semicolon in a string", "SELECT * FROM orders WHERE customer=';'"),
    ("column update_time", "SELECT update_time FROM orders"),
    ("CTE", "WITH t AS (SELECT 1) SELECT * FROM t"),
    ("schema leak", "SELECT name FROM sqlite_master"),
    ("union leak", "SELECT customer FROM orders UNION SELECT name FROM sqlite_master"),
    ("cartesian", "SELECT COUNT(*) FROM orders a, orders b, orders c"),
]
for name, sql in cases:
    try:
        guard(sql); print("%-22s PASS" % name)
    except ValueError as e:
        print("%-22s BLOCK (%s)" % (name, e))

plain select           PASS
stacked                BLOCK (multiple statements)
lower-case             BLOCK (only SELECT allowed)
word in a string       BLOCK (forbidden keyword)
semicolon in a string  BLOCK (multiple statements)
column update_time     PASS
CTE                    BLOCK (only SELECT allowed)
schema leak            PASS
union leak             PASS
cartesian              PASS


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("create table orders(id integer primary key, customer text, amount real, email text);")
con.executemany("insert into orders(customer,amount,email) values (?,?,?)",
                [("c%d" % (i % 7), i * 1.5, "u%d@x.com" % i) for i in range(300)])

def auth(action, a1, a2, db, src):
    if action == sqlite3.SQLITE_READ and a2 == "email": return sqlite3.SQLITE_DENY
    if action == sqlite3.SQLITE_READ and a1 == "sqlite_master": return sqlite3.SQLITE_DENY
    if action in (sqlite3.SQLITE_SELECT, sqlite3.SQLITE_READ, sqlite3.SQLITE_FUNCTION): return sqlite3.SQLITE_OK
    return sqlite3.SQLITE_DENY
con.set_authorizer(auth)

def run(sql, budget=None):
    ticks = [0]
    def prog():
        ticks[0] += 1
        return 1 if ticks[0] > budget else 0
    con.set_progress_handler(prog if budget else None, 1000)
    try:
        return con.execute(sql).fetchall()[:3]
    except sqlite3.Error as e:
        return "%s: %s" % (type(e).__name__, e)

print(run("SELECT customer, SUM(amount) FROM orders GROUP BY customer"))
print(run("SELECT email FROM orders"))
print(run("SELECT * FROM orders"))
print(run("SELECT name FROM sqlite_master"))
print(run("DELETE FROM orders"))
print(run("SELECT COUNT(*) FROM orders a, orders b, orders c", budget=50))

[('c0', 9481.5), ('c1', 9546.0), ('c2', 9610.5)]
DatabaseError: access to orders.email is prohibited
DatabaseError: access to orders.email is prohibited
DatabaseError: access to sqlite_master.name is prohibited
DatabaseError: not authorized
OperationalError: interrupted


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
create table orders(id integer primary key, customer text, amount real);
insert into orders values (1,'asha',100),(2,'ravi',80),(3,'asha',45),(4,'meena',200),(5,'ravi',100);
""")
def rows(sql):
    return sorted(map(lambda r: tuple(round(x, 6) if isinstance(x, float) else x for x in r), con.execute(sql).fetchall()), key=repr)
tests = [
    ("orders over 100",   "SELECT id FROM orders WHERE amount > 100",  "SELECT id FROM orders WHERE amount > 100"),
    ("total per customer", "SELECT customer, SUM(amount) FROM orders GROUP BY customer", "SELECT customer, SUM(amount) AS t FROM orders GROUP BY 1"),
    ("at least 100",      "SELECT id FROM orders WHERE amount >= 100", "SELECT id FROM orders WHERE amount > 100"),
    ("customer count",    "SELECT COUNT(customer) FROM orders", "SELECT COUNT(DISTINCT customer) FROM orders"),
]
ok = 0
for name, ref, model in tests:
    same = rows(ref) == rows(model)
    ok += same
    print("%-18s %s" % (name, "match" if same else "DIFFERENT"))
print("execution accuracy: %d/%d = %.2f" % (ok, len(tests), ok / len(tests)))

orders over 100    match
total per customer match
at least 100       DIFFERENT
customer count     DIFFERENT
execution accuracy: 2/4 = 0.50


## 12.8 Validating Fields and Counting the Cost of Pages

In [ ]:
from decimal import Decimal as D
print("float :", 0.1 + 0.2, "| equals 0.3?", 0.1 + 0.2 == 0.3)
print("decimal equals 0.3?", D("0.1") + D("0.2") == D("0.3"))

def check(inv):
    errs = []
    sub = sum(q * p for q, p in inv["items"])
    if sub != inv["subtotal"]: errs.append("subtotal %s != %s" % (inv["subtotal"], sub))
    gst = (inv["subtotal"] * inv["gst_rate"]).quantize(D("0.01"))
    if abs(gst - inv["gst"]) > D("0.01"): errs.append("gst %s != %s" % (inv["gst"], gst))
    if inv["subtotal"] + inv["gst"] != inv["total"]:
        errs.append("total %s != %s" % (inv["total"], inv["subtotal"] + inv["gst"]))
    return errs

good = {"items": [(2, D("150.00")), (1, D("2500.00"))], "subtotal": D("2800.00"),
        "gst_rate": D("0.18"), "gst": D("504.00"), "total": D("3304.00")}
bad = dict(good, gst=D("540.00"), total=D("3340.00"))
print(check(good))
print(check(bad))

float : 0.30000000000000004 | equals 0.3? False
decimal equals 0.3? True
[]
['gst 540.00 != 504.00']


In [ ]:
rows = [(0.99, 1), (0.97, 1), (0.95, 1), (0.93, 1), (0.90, 0), (0.85, 1), (0.80, 0), (0.70, 1), (0.60, 0), (0.40, 0)]
for t in (0.5, 0.75, 0.9, 0.95):
    auto = [c for conf, c in rows if conf >= t]
    print("threshold %.2f: auto %2d%%, accuracy of auto %.1f%%, sent to human %d" % (t, 100 * len(auto) / len(rows), 100 * sum(auto) / len(auto), len(rows) - len(auto)))

threshold 0.50: auto 90%, accuracy of auto 66.7%, sent to human 1
threshold 0.75: auto 70%, accuracy of auto 71.4%, sent to human 3
threshold 0.90: auto 50%, accuracy of auto 80.0%, sent to human 5
threshold 0.95: auto 30%, accuracy of auto 100.0%, sent to human 7


In [ ]:
import math
patches = math.ceil(1754 / 16) * math.ceil(1240 / 16)
tokens = patches // 4
print("patches:", patches, "tokens:", tokens)
print("image route (millions of tokens):", 100000 * tokens / 1e6)
print("text route  (millions of tokens):", 100000 * 800 / 1e6)
print("ratio:", round(tokens / 800, 2))

patches: 8580 tokens: 2145
image route (millions of tokens): 214.5
text route  (millions of tokens): 80.0
ratio: 2.68


## Chapter recap

## Practice set

**1.** (Easy) How many 16-pixel patches in a 160 by 96 image?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/genai/ch12.html